# 02 — Model training
## Smart Irrigation for Tomato (FYP)

Train and compare **XGBoost**, **Random Forest**, and **RBF SVM** that map ESP32 readings → **irrigate / don't irrigate**.

**Features (hardware + FAO-56):** temperature, humidity, soil moisture (%), pressure, VPD, ET0 proxy, heat stress, moisture deficit, dry-hot index.

**Target:** `irrigate` — tomato FAO policy. A second experiment predicts `pump_historical` to show that the old system is a soil threshold.

**Split:** stratified 80/20, 5-fold CV on the train fold. Baseline = irrigate if soil moisture < 55%.



In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.preprocess import PROCESSED_CSV, main as preprocess_main
from src.train import train

if not PROCESSED_CSV.exists():
    preprocess_main()

df = pd.read_csv(PROCESSED_CSV)
print(df["irrigate"].value_counts(normalize=True).round(3))
df[["temperature", "humidity", "soilMoisture", "vpd_kpa", "irrigate"]].head()




/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


irrigate
1    0.583
0    0.417
Name: proportion, dtype: float64


,temperature,humidity,soilMoisture,vpd_kpa,irrigate
0,29.184908,71.789699,55.092127,1.142145,1
1,33.707205,77.977391,14.032249,1.152477,1
2,24.760311,60.776282,51.405687,1.224891,1
3,32.738515,59.323543,78.831173,2.016266,0
4,25.692744,66.624914,14.871815,1.101667,1


In [2]:
# Trains all models, writes leaderboard + joblib + figures. Safe to re-run.
result = train()
print("Best model:", result["best_model"])
pd.DataFrame(result["leaderboard"])[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "cv_f1_mean"]]



random_forest             f1=0.990  auc=1.000  cv_f1=0.990
svm                       f1=0.987  auc=0.999  cv_f1=0.986
xgboost                   f1=0.993  auc=1.000  cv_f1=0.993

Best model: xgboost
Saved /Users/pramod/Desktop/Smart-Irrigation-Tomato/models/irrigation_model.joblib
Best model: xgboost


,model,accuracy,precision,recall,f1,roc_auc,cv_f1_mean
0,xgboost,0.991667,0.988669,0.997143,0.992888,0.999817,0.992501
1,random_forest,0.988333,0.994236,0.985714,0.989957,0.999531,0.989968
2,svm,0.985000,0.988539,0.985714,0.987124,0.999177,0.985717
3,soil_threshold_55pct,0.923333,0.987179,0.880000,0.930514,0.938617,NaN


## Results (already computed)

XGBoost is the production model (`models/irrigation_model.joblib`).

| Model | Accuracy | Precision | Recall | Test F1 | ROC-AUC | 5-fold CV F1 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| XGBoost | 0.992 | 0.989 | 0.997 | 0.993 | 1.000 | 0.993 |
| Random forest | 0.988 | 0.994 | 0.986 | 0.990 | 1.000 | 0.990 |
| RBF SVM | 0.985 | 0.989 | 0.986 | 0.987 | 0.999 | 0.986 |
| Soil threshold 55% | 0.923 | 0.987 | 0.880 | 0.931 | 0.939 | — |

XGBoost wins on F1 and recall. Random Forest has the highest precision. SVM is close but last among the three learned models. All three beat the 55% soil cutoff (F1 0.93 → 0.99) because they use the **climate interaction** (VPD / dry-hot index), not moisture alone.

High scores are expected: the tomato label is an agronomic function of these same sensors. The model is a **deployable approximation** of that policy, with a probability the API can threshold. That is more useful on the ESP32 than a nest of `if` statements, and it beats the single cutoff the firmware uses today (`soilMoisture <= 0`).



In [ ]:
fig_dir = ROOT / "results" / "figures"
for name in [
    "14_model_comparison.png",
    "13_f1_leaderboard.png",
    "11_roc_curves.png",
    "10_confusion_matrix.png",
    "12_feature_importance.png",
]:
    path = fig_dir / name
    if path.exists():
        display(Image(filename=str(path)))



## Feature importance

Permutation importance on the test set (drop in F1 when a column is shuffled):

| Feature | Importance |
| --- | ---: |
| dry_hot_index | 0.248 |
| soilMoisture | 0.123 |
| vpd_kpa | 0.043 |
| et0_proxy | 0.013 |
| temperature, humidity, pressure, heat_stress, moisture_deficit | ~0 (redundant given the features above) |

Temperature and humidity are **not** useless — they are already inside VPD and `dry_hot_index`. Pressure does not affect the tomato rule, matching EDA.



## Historical pump experiment

Random Forest and XGBoost reach **F1 = 1.0** when the target is `pump_historical`; SVM is just behind. That confirms EDA: the logged pump is a deterministic soil-moisture threshold. Cloning it with ML is easy and not the FYP goal. The API therefore serves the **tomato FAO** model.



In [ ]:
pump = pd.read_csv(ROOT / "results" / "pump_leaderboard.csv")
pump.sort_values("f1", ascending=False)



## Inference (ESP32 JSON)

The firmware posts `{temperature, humidity, pressure, soilMoisture, relayStatus, targetValue, device_id}`. Pressure is currently always `0`; the API substitutes the Kathmandu mean 854.27 hPa.



In [ ]:
import joblib
import pandas as pd

bundle = joblib.load(ROOT / "models" / "irrigation_model.joblib")
pipe = bundle["pipeline"]

def decide(temperature, humidity, soilMoisture, pressure=0):
    if pressure is None or pressure <= 0:
        pressure = 854.27
    frame = pd.DataFrame([{
        "temperature": temperature,
        "humidity": humidity,
        "soilMoisture": soilMoisture,
        "pressure": pressure,
    }])
    p = float(pipe.predict_proba(frame)[0, 1])
    return {"probability": round(p, 3), "relayStatus": "ON" if p >= 0.5 else "OFF", "targetValue": 100 if p >= 0.5 else 0}

print("hot, dry, dry soil", decide(32, 40, 28))
print("cool, humid, wet soil", decide(22, 75, 70))
print("warm, moderate soil (ESP32 pressure=0)", decide(29, 48, 52, 0))



## How to serve

```bash
PYTHONPATH=. uvicorn api.app:app --port 8000
# POST /predict  {"temperature": 29.2, "humidity": 48, "soilMoisture": 38.5}
```

Or `docker compose up --build`.

